# Customer Churn — Hyperparameter Tuning

## Objective

Improve the baseline customer churn models by tuning their hyperparameters using cross-validation.

## Models

- Logistic Regression
- Random Forest

## Evaluation Strategy

- 5-fold Stratified Cross-Validation
- Primary tuning metric: ROC-AUC
- Test set remains completely untouched during tuning

## Feature Set

The reduced 20-feature dataset identified during feature investigation is used.

Geographic features removed:

- City
- Lat Long
- Zip Code
- Latitude
- Longitude

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix)
import warnings
warnings.filterwarnings("ignore")

In [5]:
data = pd.read_csv("../data/processed/telco_customer_churn_cleaned.csv")
print("Dataset shape:", data.shape)
data.head()

Dataset shape: (7043, 26)


,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,CLTV,Churn Label
0,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,3239,Yes
1,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,2701,Yes
2,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,5372,Yes
3,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,...,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,5003,Yes
4,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,...,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,5340,Yes


In [6]:
target_column = "Churn Label"

X = data.drop(columns=[target_column])
y = data[target_column]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

X shape: (7043, 25)
y shape: (7043,)

Target distribution:
Churn Label
No     5174
Yes    1869
Name: count, dtype: int64


In [7]:
selected_features = ["Tenure Months","Monthly Charges","Total Charges","CLTV","Gender","Senior Citizen","Partner","Dependents","Phone Service","Multiple Lines","Internet Service","Online Security","Online Backup","Device Protection","Tech Support","Streaming TV","Streaming Movies","Contract","Paperless Billing","Payment Method"]
X = X[selected_features]

print("Selected feature count:", len(selected_features))
print("X shape:", X.shape)

Selected feature count: 20
X shape: (7043, 20)


In [8]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numerical features: 4
['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']

Categorical features: 16
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


In [9]:
def create_preprocessor():
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features)
        ]
    )

    return preprocessor

In [10]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [11]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", create_preprocessor()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

In [12]:
logistic_param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}

In [13]:
logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_param_grid,
    scoring="roc_auc",
    cv=cv_strategy,
    n_jobs=-1,
    return_train_score=True
)

print("Starting Logistic Regression tuning...")

logistic_grid.fit(X, y)

print("Logistic Regression tuning completed.")

Starting Logistic Regression tuning...


Logistic Regression tuning completed.


In [14]:
print("Best Logistic Regression parameters:")
print(logistic_grid.best_params_)

print("\nBest Cross-Validation ROC-AUC:")
print(logistic_grid.best_score_)

Best Logistic Regression parameters:
{'model__C': 10, 'model__class_weight': None}

Best Cross-Validation ROC-AUC:
0.8576756077385831


In [15]:
logistic_results = pd.DataFrame(logistic_grid.cv_results_)

logistic_results = logistic_results[
    [
        "param_model__C",
        "param_model__class_weight",
        "mean_test_score",
        "std_test_score",
        "mean_train_score"
    ]
].sort_values(
    by="mean_test_score",
    ascending=False
)

logistic_results

,param_model__C,param_model__class_weight,mean_test_score,std_test_score,mean_train_score
6,10.00,NaN,0.857676,0.008695,0.860385
4,1.00,NaN,0.857588,0.008766,0.860295
5,1.00,balanced,0.857486,0.008889,0.860262
7,10.00,balanced,0.857480,0.008784,0.860321
3,0.10,balanced,0.857071,0.009178,0.859784
2,0.10,NaN,0.856904,0.009269,0.859598
1,0.01,balanced,0.854264,0.009656,0.856684
0,0.01,NaN,0.853450,0.009722,0.855773


In [16]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", create_preprocessor()),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=1
        ))
    ]
)

In [17]:
random_forest_param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

In [18]:
random_forest_grid = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=random_forest_param_grid,
    scoring="roc_auc",
    cv=cv_strategy,
    n_jobs=-1,
    return_train_score=True
)

print("Starting Random Forest tuning...")

random_forest_grid.fit(X, y)

print("Random Forest tuning completed.")

Starting Random Forest tuning...
Random Forest tuning completed.


In [19]:
print("Best Random Forest parameters:")
print(random_forest_grid.best_params_)

print("\nBest Cross-Validation ROC-AUC:")
print(random_forest_grid.best_score_)

Best Random Forest parameters:
{'model__max_depth': 10, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 400}

Best Cross-Validation ROC-AUC:
0.8602427546886389


In [20]:
random_forest_results = pd.DataFrame(
    random_forest_grid.cv_results_
)

random_forest_results = random_forest_results[
    [
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__min_samples_split",
        "param_model__min_samples_leaf",
        "mean_test_score",
        "std_test_score",
        "mean_train_score"
    ]
].sort_values(
    by="mean_test_score",
    ascending=False
)

random_forest_results.head(10)

,param_model__n_estimators,param_model__max_depth,param_model__min_samples_split,param_model__min_samples_leaf,mean_test_score,std_test_score,mean_train_score
13,400,10,2,2,0.860243,0.005975,0.944822
15,400,10,5,2,0.860188,0.005456,0.943414
12,200,10,2,2,0.860159,0.005838,0.944484
11,400,10,5,1,0.859648,0.005658,0.948212
9,400,10,2,1,0.859364,0.006011,0.956079
14,200,10,5,2,0.859299,0.005105,0.943149
10,200,10,5,1,0.859079,0.005827,0.947815
8,200,10,2,1,0.858874,0.005866,0.955684
7,400,None,5,2,0.856018,0.005221,0.988071
23,400,20,5,2,0.855875,0.004852,0.987899


In [21]:
tuning_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Best CV ROC-AUC": [
        logistic_grid.best_score_,
        random_forest_grid.best_score_
    ]
})

tuning_summary.sort_values(
    by="Best CV ROC-AUC",
    ascending=False
)

,Model,Best CV ROC-AUC
1,Random Forest,0.860243
0,Logistic Regression,0.857676


In [22]:
tuned_models = {
    "Logistic Regression": logistic_grid.best_estimator_,
    "Random Forest": random_forest_grid.best_estimator_
}

print("Tuned models prepared:")
for name in tuned_models:
    print("-", name)

Tuned models prepared:
- Logistic Regression
- Random Forest


## Step 10 Conclusion

Hyperparameter tuning was performed for Logistic Regression and Random Forest using 5-fold Stratified Cross-Validation.

ROC-AUC was used as the primary tuning metric because the customer churn target is imbalanced and the model's probability ranking is important.

The test set was not used during hyperparameter tuning.

The tuned models will be evaluated on the untouched test set in the next stage before selecting and persisting the final customer churn model.